# Teammate #1 — Load and prepare the data

Start with the loading cell below. Then create `X`, `y`, and the data-splitting function.
The `...` inside each function is only a placeholder: replace it with your implementation.
This notebook becomes a working experiment once your team fills in the functions.

For this first experiment:

- Predict benign versus malignant. Use **1 = malignant** and **0 = benign**.
  Inspect `data.target_names`: the original scikit-learn labels use the opposite encoding.
- Use these four columns, in this order: `mean radius`, `mean texture`,
  `mean smoothness`, `mean concavity`.
- Treat radius as already available. The model may read **one** additional feature.
  This is an artificial feature budget; these descriptors are not separate clinical tests.

**Your handoff:** the four-column DataFrame `X`, the aligned label array `y`,
and a working `prepare_split` function. Agree on these formats before others start.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

In [2]:
# load the dataset, including column names and label descriptions.
data = load_breast_cancer(as_frame=True)
print('Available features:', list(data.data.columns))
print('Original target labels:', dict(enumerate(data.target_names)))

Available features: ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry', 'worst fractal dimension']
Original target labels: {0: np.str_('malignant'), 1: np.str_('benign')}


In [3]:
FEATURES = ['mean radius', 'mean texture', 'mean smoothness', 'mean concavity']
X = data.data[FEATURES].copy()

In [4]:
# y: 1 = malignant, 0 = benign
y = (data.target.to_numpy() == 0).astype(int)

In [5]:
print('X shape:', X.shape)                    # expect (569, 4)
print('y shape:', y.shape)                    # expect (569,)
print('Columns:', list(X.columns))
print('Missing values per column:\n', X.isna().sum())
counts = pd.Series(y).value_counts().sort_index()
print('Label counts (0 = benign, 1 = malignant):\n', counts)
print('Fraction malignant:', round(y.mean(), 3))
display(X.describe())

X shape: (569, 4)
y shape: (569,)
Columns: ['mean radius', 'mean texture', 'mean smoothness', 'mean concavity']
Missing values per column:
 mean radius        0
mean texture       0
mean smoothness    0
mean concavity     0
dtype: int64
Label counts (0 = benign, 1 = malignant):
 0    357
1    212
Name: count, dtype: int64
Fraction malignant: 0.373


,mean radius,mean texture,mean smoothness,mean concavity
count,569.000000,569.000000,569.000000,569.000000
mean,14.127292,19.289649,0.096360,0.088799
std,3.524049,4.301036,0.014064,0.079720
min,6.981000,9.710000,0.052630,0.000000
25%,11.700000,16.170000,0.086370,0.029560
50%,13.370000,18.840000,0.095870,0.061540
75%,15.780000,21.800000,0.105300,0.130700
max,28.110000,39.280000,0.163400,0.426800


In [6]:
SEEDS = [0, 1, 2, 3, 4]
print('Seeds:', SEEDS)

Seeds: [0, 1, 2, 3, 4]


The below function does the following:<br>
Split patients and convert features into above/below-training-median values.<br>
 Inputs:<br>
X: the four-column feature DataFrame; row order matches y. <br>
y: a 1D NumPy array of diagnosis labels (1 = malignant).<br>
seed: an integer controlling reproducible, stratified patient splits.<br>
Outputs:<br>
A dictionary with train_ids, val_ids, test_ids (integer row positions),
thresholds (a Series of training medians), and z_train, z_val, z_test.<br>
Each z is a DataFrame with the same four columns: 0 = at/below median;<br>
1 = above median. Reset each z row index so rows match y <br>[corresponding_ids].
#def prepare_split(X, y, seed): 

In [7]:
def prepare_split(X, y, seed):
    ids = np.arange(len(X))
    trainval_ids, test_ids = train_test_split(
        ids, test_size=0.20, stratify=y, random_state=seed)
    train_ids, val_ids = train_test_split(
        trainval_ids, test_size=0.25, stratify=y[trainval_ids], random_state=seed)
    train_ids, val_ids, test_ids = np.sort(train_ids), np.sort(val_ids), np.sort(test_ids)
    thresholds = X.iloc[train_ids].median()

    # 1 = above training median, 0 = at/below; reset index so rows match y[ids]
    def binarize(part_ids):
        return (X.iloc[part_ids] > thresholds).astype(int).reset_index(drop=True)

    return {
        'train_ids': train_ids,
        'val_ids': val_ids,
        'test_ids': test_ids,
        'thresholds': thresholds,
        'z_train': binarize(train_ids),
        'z_val': binarize(val_ids),
        'z_test': binarize(test_ids),
    }

In [8]:
#check dataset
rows = []
for seed in SEEDS:
    s = prepare_split(X, y, seed)
    tr, va, te = set(s['train_ids']), set(s['val_ids']), set(s['test_ids'])

    # No patient appears in two sets
    assert len(tr & va) == 0, f'seed {seed}: train and val overlap'
    assert len(tr & te) == 0, f'seed {seed}: train and test overlap'
    assert len(va & te) == 0, f'seed {seed}: val and test overlap'
    # Every patient is used exactly once
    assert len(tr | va | te) == len(X), f'seed {seed}: some patients are missing'

    rows.append({
        'seed': seed,
        'n_train': len(tr), 'n_val': len(va), 'n_test': len(te),
        'train∩val': len(tr & va), 'train∩test': len(tr & te), 'val∩test': len(va & te),
        'malignant_train': round(y[list(tr)].mean(), 3),
        'malignant_val': round(y[list(va)].mean(), 3),
        'malignant_test': round(y[list(te)].mean(), 3),
    })

display(pd.DataFrame(rows))
print('All seeds: the three sets are disjoint and together cover all', len(X), 'patients.')

,seed,n_train,n_val,n_test,train∩val,train∩test,val∩test,malignant_train,malignant_val,malignant_test
0,0,341,114,114,0,0,0,0.372,0.377,0.368
1,1,341,114,114,0,0,0,0.372,0.377,0.368
2,2,341,114,114,0,0,0,0.372,0.377,0.368
3,3,341,114,114,0,0,0,0.372,0.377,0.368
4,4,341,114,114,0,0,0,0.372,0.377,0.368


All seeds: the three sets are disjoint and together cover all 569 patients.


The dataset with IDs under **train_ids** is to be used for testing and evaluation

In [9]:
# Split patients and convert features into above/below-training-median values.
# Inputs:
#   X: the four-column feature DataFrame; row order matches y.
#   y: a 1D NumPy array of diagnosis labels (1 = malignant).
#   seed: an integer controlling reproducible, stratified patient splits.
# Outputs:
#   A dictionary with train_ids, val_ids, test_ids (integer row positions),
#   thresholds (a Series of training medians), and z_train, z_val, z_test.
#   Each z is a DataFrame with the same four columns: 0 = at/below median;
#   1 = above median. Reset each z row index so rows match y[corresponding_ids].



Use approximately **60% training / 20% validation / 20% test** patients.
Keep diagnosis proportions similar using stratification. Compute medians using
training patients only, then apply those same medians to validation and test patients.

Before handing this off, show that the three patient sets do not overlap and
explain which set trains predictions, chooses strategies, and evaluates the final result.


# Teammate #2 — Build the measurement-choice rules

A **strategy** is just a rule for choosing which extra feature to read.
For example:

- If radius is at or below its training median, read **texture**.
- If radius is above its training median, read **concavity**.
- Use radius and the chosen feature to predict benign or malignant.

Another strategy could always read texture, regardless of radius.

There are three possible extra features. Choose one for the lower-radius branch
and one for the higher-radius branch: **3 × 3 = 9 strategies** to compare.
We specify the allowed choices; validation data will select which strategy to use.

To avoid confusing number codes, represent each strategy using a dictionary with
two keys: `at_or_below` and `above`. Each value is the name of the extra feature to read.
For the example above, these values would be `mean texture` and `mean concavity`.

For each strategy, learn the final predictions from training patients. The four
possible leaves are combinations of radius bin (0/1) and acquired-feature bin (0/1).
Use the majority diagnosis at each leaf. If a leaf has no training patients, use
the overall training majority. For a tie, use 1; keep this convention fixed.

**Your handoff:** functions to train a strategy, make predictions, and draw the tree.
The predictor must only use radius and the one feature selected for that patient.


In [3]:
# Return the most common diagnosis label, using a fallback for an empty group.
# Inputs:
#   labels: a 1D array of 0/1 labels, possibly empty.
#   fallback: the label to use if labels is empty.
# Outputs:
#   One integer label, 0 or 1; use 1 if the nonempty group is tied.
def majority(labels, fallback):
    zero_count = 0
    one_count = 0

    if (len(labels) == 0):
        return fallback
    else:
    #counting majority
        for i in range(len(labels)):
            if (labels[i] == 0):
                zero_count += 1
            elif (labels[i] == 1):
                one_count += 1


        #determining majority
        if (one_count >= zero_count):
            return 1
        else:
            return 0

In [ ]:
# # testing majority function
# labels = [] - should output 0 (fallback)
# labels = [0, 1, 1, 1, 1, 0] - should output 1
# lables = [0, 0, 1, 1, 0, 1] - should output 1 (tie)
# labels = [0, 0, 0, 0, 1, 1] - should output 0
# fallback = 0
# print(majority(labels, fallback))

0


In [ ]:
# Learn the four final predictions for one measurement-choice strategy.
# Inputs:
#   z_train: Teammate #1's binarized training DataFrame.
#   y_train: the matching training labels in the same row order.
#   choices: a dictionary: at_or_below and above each name an extra feature.
# Outputs:
#   A model dictionary containing choices, leaves, and counts.
#   leaves maps (radius_bin, acquired_feature_bin) to a learned label.
#   counts maps those same four keys to their training-patient counts.
def fit_policy(z_train, y_train, choices):
    # for each prediction, you have 4 leafs (2 above radius, 2 at or below radius)
    # for each leaf, match it to the patients who fit the condition
    # then look at the numbr of benign and malignant (using majority function)
    # then repeat this for every leaf

    matches = [[], [], [], []]
    # first leaf is at or below median radius and at or below median texture
    for i in range(len(z_train)):
        curr_radius = z_train.iloc[i]["mean radius"]
        
        if (curr_radius == 0):
            curr_feature = z_train.iloc[i][choices["at_or_below"]]
            if (curr_feature == 0):
                # first leaf (0, 0)
                matches[0].append(y_train[i])
            else:
                # second leaf (0, 1)
                matches[1].append(y_train[i])
        else:
            curr_feature = z_train.iloc[i][choices["above"]]
            if (curr_feature == 0):
                # third leaf (1, 0)
                matches[2].append(y_train[i])
            else:
                # fourth leaf (1, 1)
                matches[3].append(y_train[i])
    



In [ ]:
# Follow a trained strategy and predict without reading unchosen features.
# Inputs:
#   model: a model returned by fit_policy.
#   z: a binarized DataFrame with the same four feature columns.
# Outputs:
#   Two aligned NumPy arrays: predicted 0/1 labels and chosen feature names.
#   Each patient must use radius and exactly one extra feature.
def predict_policy(model, z):
    ...  # TODO: implement this function.


In [ ]:
# Draw the measurement choices and final predictions as a small tree.
# Inputs:
#   model: a model returned by fit_policy.
#   thresholds: the training-median Series returned by prepare_split.
# Outputs:
#   A matplotlib Figure showing the radius split, extra feature on each
#   branch, and the prediction and training count at each leaf.
def draw_policy(model, thresholds):
    ...  # TODO: implement this function.


Check one patient by hand: which radius branch do they follow, which extra
feature is read, and which leaf prediction is returned?

Also change that patient's **unchosen** features. The chosen feature and prediction
must stay the same. Work with Teammate #3 to check this across all test patients.


# Teammate #3 — Compare strategies fairly

Use Teammate #1's splits and Teammate #2's prediction functions.

Compare five approaches:

1. **Initial only:** predict using radius alone.
2. **Random extra:** randomly choose one extra feature per patient.
3. **Best fixed extra:** always read the same extra feature; validation chooses which one.
4. **Adaptive extra:** validation chooses among all nine strategies from Teammate #2.
5. **All four:** use every selected feature as a reference with a larger budget.

For the initial-only and all-four references, predict the majority training label
for each observed combination of binary values. Use the same fallback/tie rule as
Teammate #2. This keeps the prediction mechanism consistent across comparisons.

Train prediction rules on training data. Choose the fixed and adaptive strategies
using validation error only. Use a fixed, documented tie-break rule. Evaluate the
selected strategies on test patients; do not change them based on the test results.

**Your handoff:** a results table, the selected strategy, and five example patient paths.


In [31]:
from itertools import product

# Learn majority-label predictions using a specified set of feature columns.
# Inputs:
#   z_train: the binarized training DataFrame.
#   y_train: the matching training labels.
#   columns: a list of feature names: radius only, or all four features.
# Outputs:
#   A model containing the chosen columns and a mapping from each possible
#   binary-value combination to a predicted label, including empty combinations.
def fit_lookup(z_train, y_train, columns):
    # TODO: implement this function.
    columns = list(columns)

    # Counting how many training patients are benign and malignant overall.
    overall_benign_count = np.sum(y_train == 0)
    overall_malignant_count = np.sum(y_train == 1)

    # Same fallback rule as 2nd teammates functions
    if overall_malignant_count >= overall_benign_count:
        fallback = 1
    else:
        fallback = 0
    
    lookup = {}

    for combination in product([0, 1], repeat=len(columns)):
        matching_rows = np.ones(len(z_train), dtype=bool)

        for column, value in zip(columns, combination):
            matching_rows &= z_train[column].to_numpy() == value

        matching_labels = y_train[matching_rows]

        count_0 = np.sum(matching_labels == 0)
        count_1 = np.sum(matching_labels == 1)

        if len(matching_labels) == 0:
            prediction = fallback
        elif count_1 >= count_0:
            prediction = 1
        else:
            prediction = 0

        lookup[combination] = int(prediction)

    return {
        "columns": columns,
        "lookup": lookup,
        "fallback": int(fallback)
    }


In [30]:
# Predict labels using only the feature columns stored in a lookup model.
# Inputs:
#   model: a model returned by fit_lookup.
#   z: a binarized feature DataFrame.
# Outputs:
#   A 1D NumPy array of predicted 0/1 labels in the original row order.
def predict_lookup(model, z):
    columns = model["columns"]
    lookup = model["lookup"]
    fallback = model["fallback"]

    predictions = []

    for _, row in z.iterrows():
        combination = tuple(row[column] for column in columns)
        prediction = lookup.get(combination, fallback)
        predictions.append(prediction)

    return np.array(predictions)


# Below is just some tests with some dummy data for the functions fit and predict. I don't have the real data in format rn so just testing a little bit.

In [29]:
import numpy as np
import pandas as pd

z_train_test = pd.DataFrame({
    "mean radius": [0, 0, 1, 1],
    "mean texture": [0, 1, 0, 1],
})

y_train_test = np.array([0, 1, 1, 1])

test_model = fit_lookup(
    z_train_test,
    y_train_test,
    ["mean radius", "mean texture"]
)

print(test_model)

test_predictions = predict_lookup(test_model, z_train_test)

print(test_predictions)

print(type(test_predictions))
print(test_predictions.shape)
print(set(test_predictions))

z_new_test = pd.DataFrame({
    "mean radius": [0, 1, 1],
    "mean texture": [1, 0, 0],
})

new_predictions = predict_lookup(test_model, z_new_test)

print(new_predictions)

z_new_with_extra_columns = z_new_test.copy()
z_new_with_extra_columns["mean smoothness"] = [0, 0, 1]
z_new_with_extra_columns["mean concavity"] = [1, 1, 0]

predictions_with_extra_columns = predict_lookup(
    test_model,
    z_new_with_extra_columns
)

print(predictions_with_extra_columns)

reversed_rows = z_new_test.iloc[::-1].reset_index(drop=True)

print(reversed_rows)
print(predict_lookup(test_model, reversed_rows))

{'columns': ['mean radius', 'mean texture'], 'lookup': {(0, 0): 0, (0, 1): 1, (1, 0): 1, (1, 1): 1}, 'fallback': 1}
[0 1 1 1]
<class 'numpy.ndarray'>
(4,)
{np.int64(0), np.int64(1)}
[1 1 1]
[1 1 1]
   mean radius  mean texture
0            1             0
1            1             0
2            0             1
[1 1 1]


In [28]:
# Calculate the fraction of predictions that are wrong.
# Inputs:
#   truth: the actual 0/1 labels.
#   pred: the predicted 0/1 labels in matching order.
# Outputs:
#   One number between 0 and 1: wrong predictions divided by all predictions.
def error_rate(truth, pred):
    return np.sum(truth != pred) / len(truth)


In [27]:
# Summarize prediction errors, including missed malignant cases.
# Inputs:
#   truth: the actual labels (1 = malignant).
#   pred: the predicted labels in matching order.
# Outputs:
#   A dictionary containing error_rate, false_negative_rate, false_positive_rate,
#   wrong_predictions, false_negatives, n_test, and n_malignant_test.
#   False-negative rate = missed malignant cases / all actual malignant cases;
#   false-positive rate = benign cases predicted malignant / all actual benign cases.
#   If a denominator is zero, report NaN rather than silently using zero.
def metrics(truth, pred):
    errors = error_rate(truth, pred)
    false_negative_rate = np.sum((truth == 1) & (pred == 0)) / np.sum(truth == 1) if np.sum(truth == 1) > 0 else np.nan
    false_positive_rate = np.sum((truth == 0) & (pred == 1)) / np.sum(truth == 0) if np.sum(truth == 0) > 0 else np.nan
    wrong_predictions = np.sum(truth != pred)
    false_negatives = np.sum((truth == 1) & (pred == 0))
    n_test = len(truth)
    n_malignant_test = np.sum(truth == 1)

    return {
        "error_rate": errors,
        "false_negative_rate": false_negative_rate,
        "false_positive_rate": false_positive_rate,
        "wrong_predictions": wrong_predictions,
        "false_negatives": false_negatives,
        "n_test": n_test,
        "n_malignant_test": n_malignant_test
    }

In [32]:
# Train, select, and evaluate the five approaches for one patient split.
# Inputs:
#   X: the original four-column feature DataFrame.
#   y: the aligned diagnosis labels.
#   seed: the random seed for reproducible splitting and random acquisition.
# Outputs:
#   A results DataFrame with one row per approach: seed, strategy, extra_features,
#   and the metrics above. Extra-feature counts are 0, 1, 1, 1, and 3 respectively.
#   Also return a details dictionary containing the split, selected model,
#   validation scores for candidate strategies, and a patient acquisition trace.
#   The trace records patient row ID, initial bin, chosen feature, acquired bin,
#   prediction, and actual label.
def run_split(X, y, seed):
    split = prepare_split(X, y, seed)

    # ids for train, validation, and test sets
    train_ids = split["train_ids"]
    val_ids = split["val_ids"]
    test_ids = split["test_ids"]

    # features for train, validation, and test sets
    z_train = split["z_train"]
    z_val = split["z_val"]
    z_test = split["z_test"]

    # labels for train, validation, and test sets
    y_train = y[train_ids]
    y_val = y[val_ids]
    y_test = y[test_ids]

    # names of all the features for each run method
    radius = "mean radius"
    extra_features = [
        "mean texture",
        "mean smoothness",
        "mean concavity",
    ]

    all_features = [
        "mean radius",
        "mean texture",
        "mean smoothness",
        "mean concavity",
    ]

    # initialize results list
    result_rows = []

    # Doing the first model (only radius)
    initial_model = fit_lookup(z_train, y_train, [radius]) # fit initial model using only radius
    initial_pred = predict_lookup(initial_model, z_test) # predict using the initial model on the test set
    initial_metrics = metrics(y_test, initial_pred) # compile metric after predicting
    result_rows.append({ # add results of this model to results list
        "seed": seed,
        "strategy": "initial_only",
        "extra_features": 0,
        **initial_metrics
    })

    # Second model (radius + 1 extra feature)
    random_models = {}

    for feature in extra_features: # fit a model for each (radius + feature) combination using the training set
        random_models[feature] = fit_lookup(z_train, y_train, [radius, feature])

    rng = np.random.default_rng(seed)
    random_choices = rng.choice(extra_features, size=len(z_test)) # randomly choose one extra feature for each patient in the test set

    predictions_by_feature = {}

    for feature in extra_features:
        predictions_by_feature[feature] = predict_lookup(random_models[feature], z_test) # predict using the model on the test set for each extra feature

    random_pred = np.empty(len(z_test), dtype=int) # initialize an empty array to store the predictions for each patient based on the randomly chosen extra feature

    for i, feature in enumerate(random_choices):
        random_pred[i] = predictions_by_feature[feature][i] # assign the prediction for each patient based on the randomly chosen extra feature

    random_metrics = metrics(y_test, random_pred) # compile metric after predicting
    result_rows.append({ # add results of this model to results list
        "seed": seed,
        "strategy": "random_extra",
        "extra_features": 1,
        **random_metrics
    })

    # Third model (radius + 1 best extra feature picked by validation)
    fixed_models = {}
    fixed_validation_scores = []

    for feature in extra_features: # fit a model for each (radius + feature) combination using the training set, compile predictions and metrics on the validation set
        model = fit_lookup(z_train, y_train, [radius, feature])
        val_pred = predict_lookup(model, z_val)
        val_error = error_rate(y_val, val_pred)
        fixed_models[feature] = model
        fixed_validation_scores.append({
            "feature": feature,
            "val_error": val_error
        })

    best_fixed_position = min(range(len(fixed_validation_scores)), key=lambda i: fixed_validation_scores[i]["val_error"]) # find the index for the best performing model based on val_error
    best_fixed_feature = fixed_validation_scores[best_fixed_position]["feature"] # find the corresponding feature for the best performing model based on val_error

    best_fixed_model = fixed_models[best_fixed_feature] # find the best model based on the best performing feature
    best_fixed_pred = predict_lookup(best_fixed_model, z_test) # predict using the best model on the test set
    best_fixed_metrics = metrics(y_test, best_fixed_pred) # compile metric after predicting

    # compile final details and add to results list
    result_rows.append({
        "seed": seed,
        "strategy": "best_fixed_extra",
        "extra_features": 1,
        **best_fixed_metrics
    })

    # Fourth model (adaptive features based on teammate 2 policies)
    policy_choices = []

    for lower_feature in extra_features: # iterate through each extra feature to be used as the "at_or_below" feature
        for higher_feature in extra_features: # iterate through each extra feature to be used as the "above" feature
            policy_choices.append({ # append a dictionary to the policy_choices list containing the "at_or_below" and "above" features for each combination of extra features
                "at_or_below": lower_feature,
                "above": higher_feature
            })

    policy_models = {}
    policy_validation_scores = []

    for choices in policy_choices: # iterate through each policy choice to fit a model and compile predictions and metrics on the validation set
        policy_key = (choices["at_or_below"], choices["above"])
        policy_model = fit_policy(z_train, y_train, choices)
        val_pred, _ = predict_policy(policy_model, z_val)
        val_error = error_rate(y_val, val_pred)
        policy_models[policy_key] = policy_model
        policy_validation_scores.append({
            "choices": choices,
            "val_error": val_error
        })


    best_policy_position = min(range(len(policy_validation_scores)), key=lambda i: policy_validation_scores[i]["val_error"]) # find the index for the best performing policy based on val_error
    best_policy_choices = policy_validation_scores[best_policy_position]["choices"] # find the corresponding choices for the best performing policy based on val_error
    best_policy_key = (best_policy_choices["at_or_below"], best_policy_choices["above"]) # find the corresponding key for the best performing policy based on val_error

    best_policy_model = policy_models[best_policy_key] # find the best model based on the best performing policy
    adaptive_pred, adaptive_features = predict_policy(best_policy_model, z_test) # predict using the best model on the test set and get the corresponding features used for each patient
    adaptive_metrics = metrics(y_test, adaptive_pred) # compile metric after predicting

    # compile final details and add to results list
    result_rows.append({
        "seed": seed,
        "strategy": "adaptive_extra",
        "extra_features": 1,
        **adaptive_metrics
    })

    # Fifth model (all four features)
    all_four_model = fit_lookup(z_train, y_train, all_features)
    all_four_pred = predict_lookup(all_four_model, z_test)
    all_four_metrics = metrics(y_test, all_four_pred)
    result_rows.append({
        "seed": seed,
        "strategy": "all_four",
        "extra_features": 3,
        **all_four_metrics
    })

    # make patient acquisition trace
    trace_rows = []

    radius_values = z_test[radius].to_numpy() # get the radius values for each patient in the test set

    for i in range(min(5, len(z_test))): # iterate through the first five patients in the test set to compile a trace of their acquisition process
        chosen_feature = adaptive_features[i] # get the chosen feature for each patient based on the adaptive policy
        acquired_bin = z_test.iloc[i][chosen_feature] # get the acquired bin for each patient based on the chosen feature
        initial_bin = radius_values[i] # get the initial bin for each patient based on the radius value

        # Determine the branch based on the initial bin value
        if initial_bin == 0:
            branch = "at_or_below"
        else:
            branch = "above"

        # compile the trace information for each patient and append it to the trace_rows list
        trace_rows.append({
            "patient_id": int(test_ids[i]),
            "initial_bin": int(initial_bin),
            "branch": branch,
            "chosen_feature": chosen_feature,
            "acquired_bin": int(acquired_bin),
            "prediction": int(adaptive_pred[i]),
            "actual_label": int(y_test[i]),
        })

    # compile final details before reutrning results and details
    details = {
        "split": split,
        "initial_model": initial_model,
        "selected_fixed_feature": best_fixed_feature,
        "fixed_validation_scores": pd.DataFrame(
            fixed_validation_scores
        ),
        "selected_policy": best_policy_choices,
        "policy_validation_scores": pd.DataFrame(
            policy_validation_scores
        ),
        "patient_trace": pd.DataFrame(trace_rows),
        "random_choices": random_choices,
    }

    results = pd.DataFrame(result_rows)

    return results, details # return the results and details for this patient split

In [33]:
# Summarize the prespecified repeated splits without treating them as independent studies.
# Inputs:
#   results: the combined results DataFrame from the five fixed seeds.
# Outputs:
#   A summary DataFrame grouped by strategy: mean error, error standard deviation,
#   mean false-negative rate, mean false-positive rate, and extra-feature count.
def summarize_runs(results):
    summary = {}

    strategies = results["strategy"].unique() # make a list of all unique strategies from results

    for strategy in strategies:
        strategy_rows = results[results["strategy"] == strategy] # filter the results for the curr strategy

        summary[strategy] = { # compile summary metrics for curr strategy
            "mean_error": strategy_rows["error_rate"].mean(),
            "error_std": strategy_rows["error_rate"].std(),
            "mean_false_negative_rate": strategy_rows["false_negative_rate"].mean(),
            "mean_false_positive_rate": strategy_rows["false_positive_rate"].mean(),
            "extra_features": strategy_rows["extra_features"].iloc[0]
        }

    summary_df = pd.DataFrame.from_dict(summary, orient="index")
    summary_df = summary_df.reset_index()
    summary_df = summary_df.rename(columns={"index": "strategy"})
    
    return summary_df # return summary DataFrame

First complete one split, then run the five seeds your team fixed in advance.
The adaptive search includes the fixed strategies: it may select a fixed one.
Do not force different branch choices or expect an improvement.

The five splits overlap. Their standard deviation describes variation; it is not
a confidence interval. Label the all-four approach as a larger-budget reference.
Save the results table and the actual chosen rules so someone else can reproduce them.


# Teammate #4 — Create incomplete records and inspect coverage

This is preparation for the **next experiment**, not a missing-data model yet.

Create two measurement groups:

- **Panel A:** radius + texture.
- **Panel B:** smoothness + concavity.

Some training patients should retain only A, others only B, and a specified
fraction should retain both. Keep the diagnosis label for every patient.
Assign panels randomly, independently of diagnosis and feature values.

Compare requested overlap fractions of **0%, 10%, 50%, and 100%**.
Count how many patients have each pair of measurements recorded together.
At zero overlap, no patient should have a feature from A together with a feature from B.

**Your handoff:** masked training/validation tables, coverage tables, and an explanation
of which measurement combinations can no longer be estimated directly.


In [2]:
# Hide one measurement panel for some patients, leaving a chosen fraction with both.
# Inputs:
#   raw: a four-column training or validation DataFrame, without diagnosis labels.
#   overlap_fraction: a number from 0 to 1 specifying the fraction retaining both panels.
#   seed: an integer controlling reproducible panel assignment.
# Outputs:
#   A copy of raw with hidden feature values replaced by NaN.
#   Preserve the original values, columns, row IDs, and row order otherwise.
#   Round the requested complete-record count and randomly assign remaining rows A or B.
# def mask_panels(raw, overlap_fraction, seed):
# TODO: implement this function.
PANEL_A = ['mean radius', 'mean texture'] #size and shade of the cell nuclei
PANEL_B = ['mean smoothness', 'mean concavity'] #how smooth and dented the nucleus outline is

def mask_panels(raw, overlap_fraction, seed): 
    rng = np.random.default_rng(seed) #random number generator where the same seed gives the same random choices
    n = len(raw) #number of patients = number of rows in the table
    n_both = int(round(overlap_fraction * n)) #number of patients with both panels
    order = rng.permutation(n) #shuffles the row positions into a random order, making the choice random
    rest = order[n_both:] #the remaining patients after the ones with both panels
    half = len(rest) // 2 #half of the remaining patients
    only_a, only_b = rest[:half], rest[half:] #split the remaining patients into two groups
    masked = raw.copy() #make a copy of the original data to modify. without the copy, the hiding would damage the orginal table
    a_cols = masked.columns.get_indexer(PANEL_A) #positions of radius, texture
    b_cols = masked.columns.get_indexer(PANEL_B) #positions of smoothness, concavity
    masked.iloc[only_a, b_cols] = np.nan #hide panel B for patients in only A
    masked.iloc[only_b, a_cols] = np.nan #hide panel A for patients in only B
    return masked #return the modified table with some hidden values

In [4]:
# Count how many patients have each pair of features observed together.
# Inputs:
#   masked: a masked four-column feature DataFrame, without diagnosis labels.
# Outputs:
#   A square DataFrame with feature names as rows and columns.
#   Entry (A, B) counts rows with both A and B present;
#   diagonal entries count how often each individual feature is present.
def joint_coverage(masked):
# TODO: implement this function.
    seen = masked.notna().astype(int) #a cell is "seen" (1) if a value is recorded, and "missing" (0) if it's NaN. this turns the table into a map of what was actually observed
    return seen.T.dot(seen) #the dot product of the transpose of the seen table and the seen table itself counts how many patients have each pair of features observed together. the diagonal entries count how often each individual feature is present

In [ ]:
#testing the functions
FEATURES = ['mean radius', 'mean texture', 'mean smoothness', 'mean concavity']
X = load_breast_cancer(as_frame=True).data[FEATURES] 
#will run this test once the import line for the first section has been run and the FEATURES variable has been created

for frac in [0, 0.1, 0.5, 1]:
    print(f'Overlap {int(frac * 100)}%')
    display(joint_coverage(mask_panels(X, frac, seed=0)))

Make masks separately for training and validation patients. Keep the test data
complete for evaluation. A future learner should only receive the masked records;
the evaluator reveals test features along the selected measurement path.

Do not reuse the complete-data medians, fitted prediction rules, or selected strategies
when implementing the missing-data experiment. Those would reveal information that
the future learner is supposed to lack. Fit preprocessing from observed training values.

Scores from Teammate #3 are **complete-training-data baseline scores**, not evidence
that a method works on these masked records. Random masking is a controlled experiment,
not a complete simulation of why clinical tests are missing.

**Optional short exercise: connect this to regret.** This is a separate invented example,
not a WDBC result. Suppose two binary test results have four equally likely combinations.
Under one possible relationship, disease occurs when results differ; under the other,
it occurs when results match. Compare strategies predicting disease for differing results,
matching results, always benign, and always malignant. Build a table with one strategy
per row and one possible relationship per column. Each entry is its prediction error.


In [6]:
# Compare each strategy with the best strategy under the same possible relationship.
# Inputs:
#   error_table: a DataFrame of error rates: rows are strategies and columns are possible relationships.
# Outputs:
#   A regret table of the same shape, each strategy's largest regret across columns,
#   and the names of all strategies tied for the smallest largest regret.
#   Regret = strategy error minus the smallest strategy error in that same column.
def calculate_regret(error_table):
# TODO: implement this function.
    regret = error_table - error_table.min(axis=0) #subtract the smallest error in each column from each entry in that column to get the regret
    worst = regret.max(axis=1) #find the largest regret for each strategy
    best = worst[np.isclose(worst, worst.min())].index.tolist() #find the strategies with the smallest largest regret and return their names as a list
    return regret, worst, best #hand back the three outputs the comments lists

In [11]:
err = pd.DataFrame(
    {'differ': [0,1,0.5,0.5],
     'match': [1,0,0.5,0.5]},
     index = ['predict differ', 'predict match', 'always benign', 'always malignant']
)
regret, worst, best = calculate_regret(err)
display(regret)
print(worst)
print('Minimax-regret strategies:', best)

,differ,match
predict differ,0.0,1.0
predict match,1.0,0.0
always benign,0.5,0.5
always malignant,0.5,0.5


predict differ      1.0
predict match       1.0
always benign       0.5
always malignant    0.5
dtype: float64
Minimax-regret strategies: ['always benign', 'always malignant']


## Bring together before the meeting

- One notebook that runs from top to bottom after you fill in the missing code.
- One selected-tree diagram and five explained patient paths.
- A baseline comparison table and the seeds/settings used.
- Coverage tables showing what disappears as joint observations become scarce.

Each teammate should review another person's section. Everyone should be able to
explain one full prediction path. This assignment builds the baseline and missingness
setup; it does not yet implement the WDBC minimax-regret selection method.
